# Ch 11 — 피마 인디언 당뇨병 예측

원본: `02_Data_preparation.py` (EDA) + `02_Pima_Indian.py` (학습)

다루는 내용:
1. 데이터 로드 + 컬럼 이름
2. EDA — head / info / describe
3. 상관관계 히트맵
4. plasma 분포 비교 (class별)
5. 모델 학습 (Dense 12-8-1)

## 0. 환경

In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import keras
from keras import Input, Sequential
from keras.layers import Dense

keras.utils.set_random_seed(0)

print("Keras:", keras.__version__)

import seaborn as sns

## 1. 데이터 로드 (컬럼 이름 부여)

In [ ]:
DATA = "../../data/pima-indians-diabetes.csv"
columns = ["pregnant", "plasma", "pressure", "thickness",
           "insulin", "BMI", "pedigree", "age", "class"]
df = pd.read_csv(DATA, names=columns)
print("shape:", df.shape)
df.head()

## 2. 기본 통계

In [ ]:
df.info()

In [ ]:
df.describe()

**관찰:** `plasma`, `pressure`, `thickness`, `insulin`, `BMI` 의 최솟값이 0인데, 이는 *결측 의미의 0*임. 진짜 0이 의학적으로 말이 안 되는 컬럼들. 책은 이 부분을 다루지 않지만, 실무에선 대치(impute)가 필요.

## 3. 상관관계 히트맵

In [ ]:
plt.figure(figsize=(9, 7))
sns.heatmap(df.corr(), annot=True, fmt=".2f",
            cmap="gist_heat", linewidths=0.1, vmax=0.5, linecolor="white")
plt.title("Pima — feature correlations")
plt.show()

## 4. plasma — 클래스별 분포

In [ ]:
g = sns.FacetGrid(df, col="class", height=3.5)
g.map(plt.hist, "plasma", bins=10, color="steelblue", edgecolor="white")
plt.show()

당뇨 발병군(`class=1`)이 비발병군(`class=0`)보다 plasma 분포가 오른쪽으로 이동.
이것이 `class` 와 `plasma` 의 양의 상관관계를 시각적으로 확인하는 방법.

## 5. 학습용 X/y 분리

In [ ]:
X = df.iloc[:, 0:8].to_numpy(dtype="float32")
y = df["class"].to_numpy(dtype="float32")
print("X:", X.shape, "y:", y.shape)
print("y 분포:", np.bincount(y.astype(int)))

## 6. 모델 — Dense(12) → Dense(8) → Dense(1)

In [ ]:
def build_model():
    return Sequential([
        Input(shape=(8,)),
        Dense(12, activation="relu"),
        Dense(8, activation="relu"),
        Dense(1, activation="sigmoid"),
    ])

keras.utils.set_random_seed(0)
model = build_model()
model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])
model.summary()

## 7. 학습

In [ ]:
hist = model.fit(X, y, epochs=200, batch_size=10, verbose=0)
print(f"final train accuracy: {hist.history['accuracy'][-1]:.4f}")

In [ ]:
plt.figure(figsize=(9, 3.5))
plt.subplot(1, 2, 1); plt.plot(hist.history["loss"]); plt.title("loss"); plt.grid(alpha=0.3)
plt.subplot(1, 2, 2); plt.plot(hist.history["accuracy"]); plt.title("accuracy"); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 8. 평가

In [ ]:
loss, acc = model.evaluate(X, y, verbose=0)
print(f"loss = {loss:.4f}, accuracy = {acc:.4f}")

## 마무리
- [ ] 0이 들어간 컬럼을 NaN으로 바꾼 뒤 평균 대치하면 정확도가 어떻게 바뀌는가?
- [ ] hidden layer 크기를 바꾸면? (12→24, 8→16)
- [ ] Train accuracy가 0.85+ 나와도, *학습 데이터 그대로 평가*한 결과라는 점 인지